![temet logo](https://temet.tng-project.org/_images/logo_lg.png)

This notebook gives a short tutorial on using [temet](https://www.github.com/dnelson/temet/) to load, analyze, and visualize TNG data.

**temet** is a Python library for the analysis of astrophysical numerical simulations.

Specifically, hydrodynamical simulations run with the [AREPO](https://wwwmpa.mpa-garching.mpg.de/~volker/arepo/) moving mesh code, as well as codes producing similarly structured outputs including [GADGET-4](https://wwwmpa.mpa-garching.mpg.de/gadget4/), [GIZMO](http://www.tapir.caltech.edu/~phopkins/Site/GIZMO.html) and [SWIFT](http://swift.dur.ac.uk/).

In addition, temet is focused on cosmological simulations for large-scale structure and galaxy formation, particularly those processed with a substructure identification algorithm (halo finder) such as `subfind`, including [Illustris](https://www.illustris-project.org), [IllustrisTNG](https://www.tng-project.org/), EAGLE, and so on. Generally speaking, any simulation data available from the TNG public data release platform (and available here in the TNG Lab) can be directly analyzed with this toolkit.

We will follow the "First Steps Walkthrough" of the [temet online documentation](https://temet.tng-project.org/), but only a very short version. It is suggested to follow that tutorial in detail.

# Installation

In [ ]:
# first, we install temet, by executing this cell.
# you must then **restart the kernel** (select Kernel -> Restart from the top menu bar) once you have installed the package.
!pip install temet

We can then import the library:

In [ ]:
import temet

# Selecting a Simulation

Most analysis is based around a “simulation parameters” object (typically called `sim` below), which specifies the simulation and snapshot of interest, among other details.

You can then select a simulation from a known list, e.g. the **TNG100-1** simulation of IllustrisTNG:

In [ ]:
sim = temet.sim(run='tng100-1')

In [ ]:
sim.simPath

# Selecting a Snapshot/Redshift

We can also (optionally) select a specific snapshot i.e. redshift. The first time you do, for a particular simulation, a cache file is made to understand all available snapshots (this will take a few seconds):

In [ ]:
sim = temet.sim(run='tng100-1', redshift=1.0)

The simulation instance `sim` has many attributes that provide metadata about the simulation, such as cosmological parameters, the unit system and unit conversion functions, the list of available snapshots, and so on. For example:

In [ ]:
sim.numPart

In [ ]:
sim.redshift

In [ ]:
sim.numSubhalos

# Loading Snapshot i.e. Particle Data

Let's switch to a smaller simulation for convenience:

In [ ]:
sim = temet.sim(run='tng50-4', redshift=0.0)

To load one or more fields of particle/cell-level data from the entire snapshot:

In [ ]:
gas_pos = sim.snapshotSubset('gas', 'Coordinates')

In [ ]:
gas_pos.shape

In [ ]:
star_pos = sim.snapshotSubset('stars', 'pos')

In [ ]:
star_pos.shape

An alternative, *suggested way to load data* is to use shorthand names for particle types, for example:

In [ ]:
temp = sim.gas('temp')

In addition to shorthand/generic aliases, many **custom fields** are defined (`temp` is an example). These calculate properties on-the-fly that are not directly available in the snapshot files. See documentation for more details.

Requesting `x = sim.gas('field')` or `y = sim.stars('field')` or `z = sim.dm('field')` or `w = sim.bhs('field')` automatically uses parallelized loading functions that will be much faster than otherwise.

We can also request only the particles/cells that belong to a particular halo or subhalo:

In [ ]:
temp = sim.gas('temp', haloID=10)
temp.mean()

In [ ]:
pos = sim.dm('pos', subhaloID=1258)
pos.shape

# Loading (Sub)Halos from the Group Catalogs

The analogous function for loading group catalog data (halo and subhalo properties) is `groupCat()`. This accepts the arguments `fieldsSubhalos` and/or `fieldsHalos`, each a list of strings of requested field names. 

The `.halos()` and `.subhalos()` shortcuts of a simulation instance can always be used instead in practice:

In [ ]:
Group_M200c = sim.halos('Group_M_Crit200')

In [ ]:
Group_M200c.shape

In [ ]:
Group_M200c.max() # code mass units i.e. 1e10 msun/h

In [ ]:
subhalos = sim.subhalos(['SubhaloSFR','SubhaloMassType'])

In [ ]:
subhalos['SubhaloSFR'].mean()

In [ ]:
# load subhalo masses of stars (particle type 4), convert units to [log msun]
mstar_logmsun = sim.units.codeMassToLogMsun(subhalos['SubhaloMassType'][:,4])

Custom fields are also defined at the group catalog level (see documentation). These will be computed, and saved to a catalog file, on-demand. For example:

In [ ]:
mstar_30pkpc = sim.subhalos('mstar_30pkpc') # msun

In [ ]:
mstar_30pkpc

*Note:* When loading snapshot or group catalog fields by their actual names, data is loaded and returned as is. In particular, the **physical units** of the resulting arrays are as in the files (typically, in the code unit system of the simulation).

In contrast, custom fields typically return arrays in useful, physical units, such as $\rm{k⁢p⁢c}$, $\rm{M_\odot}$, or $\rm{K}$.

In addition, you can load all of the available fields for a particular halo or subhalo by its ID:

In [ ]:
fof = sim.halo(10)

In [ ]:
fof

# Loading Merger Trees

The merger tree data structure can be accessed via the `mergerTree()` function. For example, to load the full main progenitor branch (MPB) of a given subhalo:

In [ ]:
subID = 250
mpb = sim.loadMPB(subID)

for i in range(mpb['SnapNum'].size):
    print(mpb['SnapNum'][i], mpb['SubfindID'][i], mpb['SubhaloMass'][i])

This loads all available fields. You can also load only a subset of fields by specifying the optional `fields` argument, which accepts a list of strings of requested field names:

In [ ]:
mpb = sim.loadMPB(subID, fields=['SnapNum','SubfindID','SubhaloMass','SubhaloSFR'])

Analogously, the main descendant branch (MDB) that goes forward in time towards $𝑧=0$ can be loaded with:

In [ ]:
mdb = sim.loadMDB(subID)

For loading a large number of merger trees at once, the `loadMPBs()` function accepts a list of subhalo IDs and returns a dictionary of arrays. It is significantly faster than calling loadMPB() multiple times.

Note: In all cases, the `treeName` parameter can be used to request a specific merger tree (it must have already been computed for the simulation).

The *SubLink trees are used by default*.


# Plotting: Galaxies (Subhalos) and Halos i.e. Catalogs

The various plotting functions in `plot.subhalos` are designed to be as general and automatic as possible. They are idea for a quick look or for exploring trends in the objects of the group catalogs, i.e. galaxies (subhalos).

Let’s examine a classic observed galaxy scaling relation: the correlation between gas-phase metallicity, and stellar mass, the “mass-metallicity relation” (MZR):

In [ ]:
# select any simulation and redshift
sim = temet.sim(run='tng100-3', redshift=1.0)

In [ ]:
temet.plot.subhalos.median(sim, 'Z_gas', 'mstar2')

We can enrich the plot in a number of ways, both by: 

* setting aesthetic options
* including additional information from the simulation. 

For example, we will shift the x-axis bounds, and also include individual subhalos as colored points, coloring based on gas fraction:

In [ ]:
temet.plot.subhalos.median(sim, 'Z_gas', 'mstar2', xlim=[8.0, 11.5], ylim=[-0.7, 0.5], scatterColor='fgas2')

The figure above highlights how lower mass galaxies have high gas fractions of nearly unity, i.e. $M_{\rm gas} \gg M_\star$ and that gas fraction slowly decreasing with stellar mass until $M_\star \sim 10^{10.5} M_\odot$. 

At this point, gas metallicity and gas fraction both start to decrease, indicating the onset of galaxy quenching due to supermassive black hole feedback.

Once you add a custom calculation for a new property of subhalos, i.e. compute a value which isn’t available by default, you can use the same plotting routines to understand how it varies across the galaxy population, and correlates with other galaxy properties.

# Plotting: Snapshots (i.e. particle/cell-level information)

Similarly, plot.snapshot provides general plotting routines focused on snapshots, i.e. particle-level data. These are also then suitable for non-cosmological simulations.

Functionality includes 1D histograms, 2D distributions, median relations, and radial profiles.

For example, we could plot the traditional 2D “phase diagram” of density versus temperature. However, we can also use any (known) quantity on either axis. Furthermore, while color can represent the distribution of mass, it can also be used to show the value of a third particle/cell property, in each pixel. Let’s look at the relationship between gas pressure and magnetic field strength at $z=0$:

In [ ]:
temet.plot.snapshot.phaseSpace2d(sim, 'gas', xQuant='pres', yQuant='bmag')

For cosmological simulations, we can also look at particle/cell properties for one or more (stacked) halos. 

For example, the radial profiles of gas temperature around Milky Way-mass halos, comparing TNG50 and TNG100:

In [ ]:
import numpy as np

sims = [temet.sim('tng50-4', redshift=0.0),
        temet.sim('tng100-3', redshift=0.0)]

subIDs = []
for sim in sims:
    m200 = sim.subhalos('m200c_log')
    ids = np.where((m200 > 12.0) & (m200 < 12.1))[0]
    print(sim, ids.size)
    subIDs.append(ids)

temet.plot.snapshot.profilesStacked1d(sims, subhaloIDs=subIDs, ptType='gas', ptProperty='temp', xlim=[0.9,2.7], ylim=[4, 6.5])

# Visualization: Galaxies and Halos

We can request a visualization of any known snapshot quantity, with a large number of options.

First, let's request TNG50-1 at $z=0$ (making the snapshot cache will take a minute):

In [ ]:
import temet
sim = temet.sim(run='tng50-2', redshift=0.0)

Then, let's make a projection of gas density on the scale of a dark matter halo:

In [ ]:
subID = sim.halo(60)['GroupFirstSub']

# overall figure config
config = {'plotStyle': 'edged', 'saveFilename':None}

# common variables shared between all panels
common = {'sP': sim, 'partType': 'gas', 'partField':'coldens_msunkpc2', 'labelHalo':'mhalo,mstar'}

# define panels
panels = [{'subhaloInd':subID}]

temet.vis.halo.renderSingleHalo(panels, config, common)

The images (or “panels”) in a figure are always specified by panels, a list of dictionaries.
* Each dictionary specifies the options for that particular panel.
* Any option not specified will take the common value from common, if present, or else a default value.

The particular halo to visualize is chosen with the `subhaloInd` parameter. In the case above, this is set to be the central subhalo of Halo ID 60 (chosen at random).

We can also visualize an entire box:

In [ ]:
sim = temet.sim(run='tng100-3', redshift=0.0)

config = {'saveFilename':None}
panels = [{'sP':sim, 'partType':'gas', 'partField':'xray_lum'}]

temet.vis.box.renderBox(panels, config)

The `temet.vis.halo.renderSingleHalo()` and `temet.vis.box.renderBox()` functions both accept dozens of arguments that control the details of the visualization.

These can vary for each and every panel.

The simulation `sP` can also vary by panel, allowing for comparisons between simulations and/or snapshots.

In addition, there are also global options that apply to all panels. See the [Visualization documentation](https://temet.tng-project.org/visualization.html) for details.

# Defining Custom Fields

So far we have been exclusively exploring and visualizing existing data directly available in the catalogs (e.g. galaxy stellar mass, galaxy size) or snapshots (e.g. gas density, magnetic field strength).

Often, additional quantities of interest can be directly derived from these existing fields. For instance, while gas temperature is often not stored in snapshots, it can be derived from internal energy.

We can define new “custom fields”. These are computed on-the-fly, when requested. To do so, a Python function containing the operational definition should be decorated as:

In [ ]:
from temet.load.groupcat import catalog_field

@catalog_field
def my_ssfr(sim, field):
    """ Galaxy specific star formation rate, instantaneous (total subhalo SFR normalized by fiducial stellar mass)."""
    sfr = sim.subhalos("SubhaloSFR") # Msun/yr

    mstar = sim.subhalos("SubhaloMassInRadType")[:, sim.ptNum("stars")] # code mass
    mstar_msun = sim.units.codeMassToMsun(mstar)

    ssfr = sfr / mstar_msun

    return ssfr

my_ssfr.label = r"My sSFR"
my_ssfr.units = r"$\rm{yr^{-1}}$"
my_ssfr.limits = [-13.3, -8.0]
my_ssfr.log = True

Here we have defined a new field, `my_ssfr`, the galaxy specific star formation rate (sSFR), which is the total instantaneous star formation rate (measured within the entire subhalo) divided by stellar mass (measured within twice the stellar half mass radius). The four lines at the end assign metadata and plot units: a label, the units of the return, suggested plot limits, and whether or not the quantity should be plotted on logarithmic axes.

We can then request a plot that includes this new field:

In [ ]:
sim = temet.sim(run='tng300-3', redshift=0.1)

temet.plot.subhalos.histogram2d(sim, 'my_ssfr', 'mstar2', cQuant='fgas2', ylim=[-11.5, -8], cRel=[-0.1,0.1,True])

This is an example of a custom field defined for the group catalog, i.e. for subhalos.

Similarly, custom fields can be defined for snapshot quantities, i.e. fields of particles/cells, with the `temet.load.snapshot.snap_field` decorator.

# Next Steps

You should have a feel for how to get started with **temet** now.

The full [Getting Started Walkthrough](https://temet.tng-project.org/) contains a more in-depth tutorial, and the documentation contains more details on key topics:

* loading data.
* plotting.
* visualization.
* physical quantities and defining custom fields.
* generating catalogs.
* advanced topics.
